# Baseline Evaluation (Colab)

Colab port of `kaggle_baselines.ipynb`. Runs the physics and learned baselines against the IQ L=50 dataset; all baselines are compared on the same held-out test split.

**Difference from the Kaggle notebook:** the dataset files are pulled from the **Kaggle dataset `noso0s0n/iql50`** via the Kaggle API (instead of being mounted as a notebook input), and checkpoints + plots persist on a **mounted Google Drive** (instead of the rclone secret). The first code cell needs Colab Secrets `KAGGLE_USERNAME` and `KAGGLE_KEY` (from your `kaggle.json` API token).

**Metrics** (accumulated globally across the whole test set, not averaged per-batch; see `Baselines/metrics.py`):
- MSLE, R² (raw), R² (log1p), CPU µs/atom — summary bar chart
- Per-q R², per-q %err, and an aggregated per-q summary — does performance hold at high q?
- Kratky log1p(I(q)) overlay — curve shape, not just scale
- **Per-molecule** signed-residual and MSLE distributions (with skew) — the honest error distribution, not the per-q mean proxy
- Error and signed-residual vs atom count (+ scaling-slope fit) — size-scaling and directional bias

**µs/atom caveat:** it is measured with `time.process_time()` (CPU only, no `cuda.synchronize`), so the GPU count is irrelevant — 1 GPU on Colab changes nothing. But it *does* depend on the CPU of the machine, so **do not compare µs/atom across Kaggle and Colab**; only compare numbers produced on one box.

**Checkpointing / resume:** results are written to `CKPT_DIR` on Drive after every baseline. Rerun top-to-bottom to resume — completed baselines are skipped.

In [ ]:
# ── data source: Kaggle dataset  |  checkpoints + plots: Google Drive ───────
# The two dataset files (HDF5 + encoding DB) are pulled from the Kaggle dataset
# noso0s0n/iql50 via the Kaggle API, so no manual upload to Drive is needed.
# Drive is still mounted only to persist checkpoints/plots across sessions.
#
# Kaggle auth: kaggle.com -> Settings -> "Create New API Token" downloads
# kaggle.json ({"username":..., "key":...}). Put those two values into Colab
# Secrets (🔑 in the left sidebar) as KAGGLE_USERNAME and KAGGLE_KEY, and toggle
# "Notebook access" on for this notebook.
import os, subprocess, sys

from google.colab import userdata
try:
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY']      = userdata.get('KAGGLE_KEY')
except Exception as e:
    raise RuntimeError(
        'Missing Kaggle credentials. Add Colab Secrets KAGGLE_USERNAME and '
        'KAGGLE_KEY (values from your kaggle.json API token) and enable notebook '
        'access for this notebook (🔑 in the left sidebar).'
    ) from e

KAGGLE_DATASET = 'noso0s0n/iql50'
DATA_DIR  = '/content/iql50'
HDF5_PATH = f'{DATA_DIR}/I(q)L50.h5'
DB_NAME   = f'{DATA_DIR}/iq_train_set-ENCODING.sqlite3'

# download once per session (the container's /content is wiped between sessions)
if not (os.path.exists(HDF5_PATH) and os.path.exists(DB_NAME)):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kaggle'], check=True)
    os.makedirs(DATA_DIR, exist_ok=True)
    subprocess.run(
        ['kaggle', 'datasets', 'download', '-d', KAGGLE_DATASET, '-p', DATA_DIR, '--unzip'],
        check=True,
    )

# checkpoints + plots persist on Drive across sessions
from google.colab import drive
drive.mount('/content/drive')
CKPT_DIR = '/content/drive/MyDrive/APS360/baselines_ckpts'

REPO      = '/content/APS360'
N_BUCKETS = 10   # atom-size buckets to evaluate, randomly sampled (< 57 for speed)

assert os.path.exists(HDF5_PATH), f'HDF5 not found at {HDF5_PATH} — Kaggle download failed?'
assert os.path.exists(DB_NAME),   f'encoding DB not found at {DB_NAME} — Kaggle download failed?'
os.makedirs(CKPT_DIR, exist_ok=True)
print('paths OK')

In [ ]:
import subprocess, sys

# install deps
%pip install -q xraydb beartype jaxtyping hdf5plugin h5py "scikit-learn>=1.3"   # >=1.3 for sklearn.cluster.HDBSCAN

# plots (Baselines/metrics.py) render text through real LaTeX (xelatex) with
# JuliaMono — classic latex+dvipng can't load an arbitrary system font, only
# xelatex/lualatex + fontspec can. This apt step is the slow part (~2–3 min).
!sudo apt-get update -q && sudo apt-get install -y -q texlive-xetex texlive-latex-recommended texlive-fonts-recommended
!mkdir -p ~/.fonts && curl -fsSL https://github.com/cormullion/juliamono/releases/latest/download/JuliaMono-ttf.tar.gz \
    | tar -xz -C ~/.fonts && fc-cache -f ~/.fonts

# clone repo
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', 'https://github.com/noshou/APS360.git', REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'pull'], check=True)

sys.path.insert(0, REPO)

In [ ]:
# ── checkpointing to the mounted Drive ── (plain file copy; no rclone needed) ─
import json
from Baselines.metrics import EvalResult

_CKPT = os.path.join(CKPT_DIR, 'baselines_results.json')

def load_checkpoint() -> dict:
    """Return name -> EvalResult for baselines already completed on Drive.

    Returns the full result (per-q arrays + per-molecule distributions), so a
    resumed baseline still has plot data this session. Entries in the old
    pre-EvalResult schema (bare list) are dropped so they simply re-run.
    """
    if not os.path.exists(_CKPT):
        print('No existing checkpoint on Drive — fresh run.')
        return {}
    with open(_CKPT) as f:
        raw = json.load(f)
    data, stale = {}, []
    for name, v in raw.items():
        (data.__setitem__(name, EvalResult.from_json(v)) if isinstance(v, dict)
         else stale.append(name))
    if stale:
        print(f'Dropping {len(stale)} old-schema entry(ies), will re-run: {stale}')
    print(f'Resumed {len(data)} completed baseline(s): {list(data.keys())}')
    return data

def save_checkpoint(results: dict) -> None:
    """Write the checkpoint to Drive. Call after every baseline.

    Writes to a temp file then renames, so a session that dies mid-write never
    leaves a truncated JSON that would fail to load on the next resume.
    """
    tmp = _CKPT + '.tmp'
    with open(tmp, 'w') as f:
        json.dump({name: r.to_json() for name, r in results.items()}, f, indent=2)
    os.replace(tmp, _CKPT)
    print(f'checkpoint saved to Drive ({len(results)} baseline(s))')

In [ ]:
import h5py, hdf5plugin, torch
from Preprocess.encode import Encoding
from ScatterNet.utils.config import DEFAULT_BUCKETS

print('Loading encoding DB...')
enc = Encoding(DB_NAME, HDF5_PATH)
print(f'  {enc.count():,} molecules  |  max atoms: {enc._max}')

with h5py.File(HDF5_PATH, 'r') as f:
    q_grid = f['q_grid'][()]
    energy = float(f.attrs.get('energy', 10000.0))

q_grid = torch.from_numpy(q_grid).float()
print(f'  q_grid: {len(q_grid)} points  |  energy: {energy} eV')

In [ ]:
import random, time
from ScatterNet.batching import Batcher, Batch
from torch.utils.data import DataLoader

BUCKET_SAMPLE_SEED = 3092983   # deterministic; keep == kaggle_baselines.ipynb to sample the same buckets
LOADER_WORKERS = 2              # Colab CPUs are modest; 2 parallel HDF5 readers is plenty

def _first(x):
    return x[0]

def _materialize(dataset, name, num_workers=LOADER_WORKERS):
    """Read every batch out of `dataset` once and cache it as a plain list, so the
    5 fits + 8 evaluates below share one pass over the data instead of ~13."""
    loader = DataLoader(dataset, batch_size=1, collate_fn=_first, num_workers=num_workers)
    out, t0 = [], time.time()
    for i, batch in enumerate(loader):
        out.append(batch)
        print(f'\r  materializing {name}: {i+1}/{len(dataset)}  ({time.time()-t0:.0f}s elapsed)',
              end='', flush=True)
    print()
    return out

eval_buckets = sorted(
    random.Random(BUCKET_SAMPLE_SEED).sample(DEFAULT_BUCKETS, min(N_BUCKETS, len(DEFAULT_BUCKETS)))
)

batcher = Batcher(
    hdf5_db        = HDF5_PATH,
    enc            = enc,
    batches        = eval_buckets,
    seed           = 42,
    atom_size_ceil = 6046,
)
train_set, _, test_set = batcher.get_sets()

test_loader  = _materialize(test_set,  'test set')
train_loader = _materialize(train_set, 'train set')
print(f'Test batches: {len(test_loader)}  |  Train batches: {len(train_loader)}')

In [ ]:
from Baselines.metrics import evaluate as _evaluate

def evaluate(baseline, loader, name):
    """Evaluate a baseline, print its headline metrics, return the full EvalResult."""
    result = _evaluate(baseline, loader, q_grid, name)
    print(f'{name:<30s}  MSLE={result.msle:.4f}  R²(raw)={result.r2_raw:.4f}  '
          f'R²(log1p)={result.r2_log1p:.4f}  {result.us_per_atom:.2f} μs/atom')
    return result

In [ ]:
sys.path.insert(0, f'{REPO}/Baselines/physics-benchmarks')
sys.path.insert(0, f'{REPO}/Baselines/learned-benchmarks')

from rg import RgBaseline, GuinierPorodBaseline
from atom_count import AtomCountBaseline
from pair_peak import BinnedDebyeBaseline
from saxs_est import SaxsEstBaseline

print('=== Physics Baselines ===')
results = load_checkpoint()

# factories, not instances: construction/fit is deferred until we know a
# baseline isn't already in `results`, so a completed one never re-fits on resume
physics_baselines = [
    ('Guinier (Rg)',   lambda: RgBaseline(q_grid, energy)),
    ('Guinier-Porod',  lambda: GuinierPorodBaseline(q_grid, energy)),
    ('Atom Count',     lambda: AtomCountBaseline().fit(train_loader)),
    ('Binned Debye',   lambda: BinnedDebyeBaseline(q_grid, energy)),
    ('SAXS propoEst',  lambda: SaxsEstBaseline(q_grid, energy, method='propo', e=0.380)),
    ('SAXS stratEst',  lambda: SaxsEstBaseline(q_grid, energy, method='strat', a=0.6)),
]

for name, make_baseline in physics_baselines:
    if name in results:
        print(f'{name:<30s}  (skipped, resumed from checkpoint)')
        continue
    results[name] = evaluate(make_baseline(), test_loader, name)
    save_checkpoint(results)

In [ ]:
from torch_mlp import TorchMlp
from linsvm import Linsvm
from nearest_neighbour import NNBaseline
from hdbscan import Hdbscan

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print('=== Learned Baselines ===')

learned_baselines = [
    ('MLP',                lambda: TorchMlp().fit(train_loader)),
    ('Linear SVM',         lambda: Linsvm().fit(train_loader)),
    ('Nearest Neighbour',  lambda: NNBaseline(q_grid, energy).fit(train_loader)),
    ('HDBSCAN',            lambda: Hdbscan(q_grid, energy)),   # analytical, no .fit() needed
]

for name, make_baseline in learned_baselines:
    if name in results:
        print(f'{name:<30s}  (skipped, resumed from checkpoint)')
        continue
    results[name] = evaluate(make_baseline(), test_loader, name)
    save_checkpoint(results)

In [ ]:
_known = {n for n, _ in physics_baselines} | {n for n, _ in learned_baselines}
_stale = sorted(n for n in results if n not in _known)
if _stale:
    print(f'Dropping stale checkpoint entries no longer in the baseline list: {_stale}')
    for n in _stale:
        del results[n]
    save_checkpoint(results)

print('\n=== Summary ===')
print(f"{'Baseline':<30s}  {'MSLE':>8s}  {'R²(raw)':>10s}  {'R²(log1p)':>12s}  {'μs/atom':>10s}  {'resid skew':>11s}")
print('-' * 90)
for name, r in sorted(results.items(), key=lambda x: x[1].msle):
    skew = (r.resid_stats or {}).get('skew', float('nan'))
    print(f'{name:<30s}  {r.msle:>8.4f}  {r.r2_raw:>10.4f}  {r.r2_log1p:>12.4f}  {r.us_per_atom:>10.2f}  {skew:>11.3f}')

In [ ]:
from Baselines.metrics import run_all_plots

PLOTS_DIR = os.path.join(CKPT_DIR, 'baseline_plots')
written = run_all_plots(list(results.values()), q_grid, PLOTS_DIR)
print(f'wrote {len(written)} plot(s) to Drive:', PLOTS_DIR, *written, sep='\n  ')

Plots written to `CKPT_DIR/baseline_plots` on Drive: `summary`, `per_q_r2`, `per_q_percent_error`, `per_q_summary`, `error_vs_atom_count`, `residual_vs_atom_count`, plus per-baseline `kratky_*`, `residual_histogram_*` (per-molecule signed residual + skew), and `per_molecule_msle_*`. See `Baselines/metrics.py` for what each shows.